In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logomaker
from collections import Counter

In [ ]:
ticks = {
    411:np.array([1,2,3,4,5]),
    27:np.array([1,3,6,9,12,15,18]),
    5:np.array([1,3,5,7,9,11])
}

aas = list('ACDEFGHIKLMNPQRSTVWY-')
for clone_id in [411, 27, 5]:
    CDR3s = [cdr.strip() for cdr in open(f'{clone_id}_aligned.fasta','r').readlines()[1::2]]
    
    df = pd.DataFrame(index=range(len(CDR3s[0])),columns=aas)
    df.loc[:,:] = 0
    for cdr in CDR3s:
        for i,aa in enumerate(cdr):
            df.loc[i,aa] += 1
    df /= len(CDR3s)
    df = df.astype(float)
    del df['-']
    df = df[df.sum(axis=1)>0.001].reset_index(drop=True)
    
    entropy = -(df*np.log2(df+(1e-3/len(CDR3s)))).sum(axis=1)
    correction = 19/(np.log(2)*2*len(CDR3s))
    information = np.log2(20) - (entropy + correction)
    height = df.mul(information,axis='index')
    
    plt.figure()
    logomaker.Logo(height)
    plt.xticks(ticks[clone_id]-1,ticks[clone_id],fontsize=24)
    plt.yticks([])
    
    plt.savefig(f'{clone_id}_logo.png',dpi=300,bbox_inches='tight')